# 2. Partición común y preprocesamiento con scikit-learn

Secciones 3.1 y 3.2 del enunciado.

* La partición 80/20 estratificada se define **una sola vez** (semilla 42) y se guarda en
  `data/particion_train_test.parquet` con las columnas `id` y `split`. PySpark leerá
  exactamente ese archivo (no se usa `randomSplit`).
* Todos los transformadores que aprenden parámetros (imputador, escalador, codificador y la
  agrupación de categorías raras) se ajustan **solo con train** y luego se aplican a test.
* El tiempo total de preprocesamiento (lectura + construcción de variables + partición +
  transformación) se mide para compararlo con PySpark.

In [1]:
import sys  # Permite modificar la ruta de búsqueda de módulos
sys.path.insert(0, "..")  # Agrega la raíz del proyecto
import json  # Escritura de metadatos
import joblib  # Serialización de los transformadores ajustados
import numpy as np  # Cálculo numérico
import pandas as pd  # Manipulación de datos
from sklearn.impute import SimpleImputer  # Imputación de nulos
from sklearn.model_selection import train_test_split  # Partición estratificada
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # Codificación y escalado
from src import config as cfg  # Configuración central
from src import preparacion as prep  # Construcción de variables con pandas
from src.utilidades import actualizar_json, cronometro, info_hardware  # Utilidades

tiempos = {}  # Diccionario donde se acumulan los tiempos de cada etapa
print(info_hardware())  # Hardware utilizado

{'sistema': 'Windows 10', 'procesador': 'AMD64 Family 25 Model 117 Stepping 2, AuthenticAMD', 'nucleos_fisicos': 6, 'nucleos_logicos': 12, 'ram_total_gb': 13.8, 'ram_disponible_gb': 7.3, 'python': '3.9.25'}


## Lectura y construcción de variables

Se leen del CSV completo (todas las filas) solo las columnas necesarias.

In [2]:
with cronometro(tiempos, "lectura_csv"):  # Mide el tiempo de lectura
    df_raw = pd.read_csv(cfg.RUTA_CSV, usecols=prep.COLUMNAS_ORIGINALES, low_memory=False)  # Lee todas las filas del CSV
with cronometro(tiempos, "construccion_variables"):  # Mide el tiempo de construcción de variables
    df = prep.construir_variables(df_raw).drop(columns=["issue_d"])  # Filtra estados finales y deriva variables
del df_raw  # Libera memoria
df.to_parquet(cfg.RUTA_MODELADO, index=False)  # Guarda el dataset de trabajo en el espacio original (lo usa LIME)
print(df.shape, tiempos)  # Dimensión y tiempos parciales
df.head()  # Primeras filas

(1345310, 30) {'lectura_csv': 21.665446043014526, 'construccion_variables': 2.339240074157715}


,id,loan_amnt,term_meses,int_rate,installment,annual_inc,dti,fico_range_low,fico_range_high,emp_length_anios,...,antiguedad_credito_meses,grade,sub_grade,home_ownership,verification_status,purpose,addr_state,initial_list_status,application_type,default
0,68407277,3600.0,36.0,13.99,123.03,55000.0,5.91,675.0,679.0,10.0,...,148.0,C,C4,MORTGAGE,Not Verified,debt_consolidation,PA,w,Individual,0
1,68355089,24700.0,36.0,11.99,820.28,65000.0,16.06,715.0,719.0,10.0,...,192.0,C,C1,MORTGAGE,Not Verified,small_business,SD,w,Individual,0
2,68341763,20000.0,60.0,10.78,432.66,63000.0,10.78,695.0,699.0,10.0,...,184.0,B,B4,MORTGAGE,Not Verified,home_improvement,IL,w,Joint App,0
3,68476807,10400.0,60.0,22.45,289.91,104433.0,25.37,695.0,699.0,3.0,...,210.0,F,F1,MORTGAGE,Source Verified,major_purchase,PA,w,Individual,0
4,68426831,11950.0,36.0,13.44,405.18,34000.0,10.20,690.0,694.0,4.0,...,338.0,C,C3,RENT,Source Verified,debt_consolidation,GA,w,Individual,0


## 3.1 Partición común 80/20 estratificada

In [3]:
with cronometro(tiempos, "particion"):  # Mide el tiempo de la partición
    ids_train, ids_test = train_test_split(  # Divide los identificadores
        df[cfg.COL_ID],  # Identificador único de cada préstamo
        test_size=cfg.PROPORCION_TEST,  # 20 % para prueba
        stratify=df[cfg.COL_TARGET],  # Estratifica por clase
        random_state=cfg.SEMILLA,  # Semilla fija
    )
    particion = pd.DataFrame({cfg.COL_ID: df[cfg.COL_ID]})  # Tabla con todos los id
    particion["split"] = np.where(particion[cfg.COL_ID].isin(set(ids_test)), "test", "train")  # Asigna train o test
    particion.to_parquet(cfg.RUTA_PARTICION, index=False)  # Guarda la asignación en Parquet
resumen = particion.merge(df[[cfg.COL_ID, cfg.COL_TARGET]], on=cfg.COL_ID).groupby("split")[cfg.COL_TARGET].agg(["count", "mean"])  # Tamaño y tasa de default por conjunto
resumen.columns = ["n", "tasa_default"]  # Nombres descriptivos
resumen  # Muestra que la estratificación conserva la proporción de clases

,n,tasa_default
split,,
test,269062,0.199627
train,1076248,0.199626


In [4]:
es_test = df[cfg.COL_ID].isin(set(ids_test)).to_numpy()  # Máscara de filas de test
train = df.loc[~es_test].reset_index(drop=True)  # Conjunto de entrenamiento
test = df.loc[es_test].reset_index(drop=True)  # Conjunto de prueba
print(f"train: {len(train):,}  test: {len(test):,}")  # Tamaños

train: 1,076,248  test: 269,062


## 3.2 Preprocesamiento (ajustado solo con train)

1. Transformaciones deterministas: `log1p` en `annual_inc` y `revol_bal` (asimetría muy alta)
   e indicadores de nulo de `mths_since_last_delinq`, `emp_length_anios` y `mort_acc` (nulos no MCAR, ver EDA).
2. Categóricas: nulos → `DESCONOCIDO`; categorías con < 1 % en train → `OTROS`; `OneHotEncoder`.
3. Numéricas: imputación con la mediana de train y `StandardScaler` (necesario para la
   regresión logística y LinearSVC; los árboles son invariantes a esta transformación).

Se excluyen `fico_range_low`, `installment` y `sub_grade` por multicolinealidad (EDA).

In [5]:
with cronometro(tiempos, "ajuste_y_transformacion"):  # Mide el tiempo de ajuste y transformación
    frecuentes = prep.categorias_frecuentes(train, cfg.CATEGORICAS_MODELO)  # Categorías frecuentes aprendidas en train
    train_t = prep.transformar_numericas(prep.agrupar_categorias(train, frecuentes))  # Aplica reglas a train
    test_t = prep.transformar_numericas(prep.agrupar_categorias(test, frecuentes))  # Aplica las mismas reglas a test
    numericas = cfg.NUMERICAS_MODELO + cfg.INDICADORES_NULO  # Columnas numéricas finales (incluye indicadores de nulo)
    imputador = SimpleImputer(strategy="median").fit(train_t[numericas])  # Aprende la mediana de cada numérica en train
    escalador = StandardScaler().fit(imputador.transform(train_t[numericas]))  # Aprende media y desviación en train
    codificador = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32).fit(train_t[cfg.CATEGORICAS_MODELO])  # Aprende las categorías en train

    def transformar(d):  # Aplica los transformadores ajustados a un conjunto
        num = escalador.transform(imputador.transform(d[numericas])).astype(np.float32)  # Imputa y escala las numéricas
        cat = codificador.transform(d[cfg.CATEGORICAS_MODELO])  # Codifica las categóricas
        return np.hstack([num, cat])  # Une ambas partes en una matriz densa

    X_train = transformar(train_t)  # Matriz de entrenamiento
    X_test = transformar(test_t)  # Matriz de prueba
nombres = numericas + codificador.get_feature_names_out(cfg.CATEGORICAS_MODELO).tolist()  # Nombres de las columnas finales
print("X_train:", X_train.shape, " X_test:", X_test.shape)  # Dimensiones
print("Categorías conservadas:", json.dumps(frecuentes, ensure_ascii=False)[:1500])  # Categorías tras agrupar raras
tiempos  # Tiempos de cada etapa

X_train: (1076248, 77)  X_test: (269062, 77)
Categorías conservadas: {"grade": ["A", "B", "C", "D", "E", "F"], "home_ownership": ["MORTGAGE", "OWN", "RENT"], "verification_status": ["Not Verified", "Source Verified", "Verified"], "purpose": ["car", "credit_card", "debt_consolidation", "home_improvement", "major_purchase", "medical", "other", "small_business"], "addr_state": ["AL", "AZ", "CA", "CO", "CT", "FL", "GA", "IL", "IN", "LA", "MA", "MD", "MI", "MN", "MO", "NC", "NJ", "NV", "NY", "OH", "OR", "PA", "SC", "TN", "TX", "VA", "WA", "WI"], "initial_list_status": ["f", "w"], "application_type": ["Individual", "Joint App"]}


{'lectura_csv': 21.665446043014526,
 'construccion_variables': 2.339240074157715,
 'particion': 0.5838937759399414,
 'ajuste_y_transformacion': 4.716569900512695}

In [6]:
pd.DataFrame(X_train[:5], columns=nombres)  # Vista de las primeras filas ya transformadas

,loan_amnt,term_meses,int_rate,annual_inc,dti,fico_range_high,emp_length_anios,delinq_2yrs,inq_last_6mths,open_acc,...,addr_state_SC,addr_state_TN,addr_state_TX,addr_state_VA,addr_state_WA,addr_state_WI,initial_list_status_f,initial_list_status_w,application_type_Individual,application_type_Joint App
0,-1.241053,-0.563901,0.156822,-0.292642,-1.092169,-0.664946,1.126204,-0.361833,0.367422,-0.838320,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,0.639861,1.773359,-0.515998,-0.054400,-0.662392,-0.037437,1.126204,-0.361833,-0.698334,-1.020825,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
2,-0.283393,-0.563901,0.041541,-1.136426,-0.713577,-0.194314,-0.549122,-0.361833,-0.698334,-1.203329,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,0.639861,-0.563901,-0.853456,1.787357,-0.319099,-0.508069,1.126204,-0.361833,-0.698334,0.074202,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,0.639861,-0.563901,-0.995985,0.471055,-0.059644,0.276318,1.126204,0.776533,-0.698334,-0.655816,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


In [7]:
pd.DataFrame({"media_train": X_train[:, :len(numericas)].mean(axis=0), "desv_train": X_train[:, :len(numericas)].std(axis=0),  # Verifica que train quede estandarizado
              "media_test": X_test[:, :len(numericas)].mean(axis=0)}, index=numericas)  # En test la media es ≈ 0 pero no exacta (no hay fuga)

,media_train,desv_train,media_test
loan_amnt,7.921913e-06,0.999943,-0.000559
term_meses,2.709119e-05,0.997800,-0.000083
int_rate,3.164150e-05,0.999747,-0.002307
annual_inc,1.627299e-06,0.999656,-0.000004
dti,-1.962163e-08,0.999688,-0.001487
fico_range_high,1.558067e-05,1.000371,-0.001259
emp_length_anios,-3.991100e-06,0.998668,0.001733
delinq_2yrs,-1.366328e-04,0.998681,-0.000323
inq_last_6mths,-2.800880e-04,1.001737,-0.000885
open_acc,8.545276e-06,0.999256,0.000091


## Guardado de los datos preprocesados

In [8]:
np.save(cfg.DIR_SKLEARN_PREP / "X_train.npy", X_train)  # Matriz de entrenamiento
np.save(cfg.DIR_SKLEARN_PREP / "X_test.npy", X_test)  # Matriz de prueba
np.save(cfg.DIR_SKLEARN_PREP / "y_train.npy", train[cfg.COL_TARGET].to_numpy())  # Etiquetas de entrenamiento
np.save(cfg.DIR_SKLEARN_PREP / "y_test.npy", test[cfg.COL_TARGET].to_numpy())  # Etiquetas de prueba
np.save(cfg.DIR_SKLEARN_PREP / "id_train.npy", train[cfg.COL_ID].to_numpy())  # Identificadores de entrenamiento
np.save(cfg.DIR_SKLEARN_PREP / "id_test.npy", test[cfg.COL_ID].to_numpy())  # Identificadores de prueba
joblib.dump({"frecuentes": frecuentes, "imputador": imputador, "escalador": escalador,  # Transformadores ajustados (para LIME)
             "codificador": codificador, "numericas": numericas, "nombres": nombres},  # Metadatos de columnas
            cfg.DIR_SKLEARN_PREP / "transformadores.joblib")  # Archivo de salida
tiempos["total"] = sum(tiempos.values())  # Tiempo total de preprocesamiento
actualizar_json(cfg.RUTA_TIEMPOS_PREP, "sklearn", tiempos)  # Guarda los tiempos para la comparación final
print({k: round(v, 1) for k, v in tiempos.items()})  # Muestra los tiempos en segundos

{'lectura_csv': 21.7, 'construccion_variables': 2.3, 'particion': 0.6, 'ajuste_y_transformacion': 4.7, 'total': 29.3}
